In [ ]:
# import pandas as pd
# from pathlib import Path
# import selfies as sf

# ROOT = Path("..")   # adjust if needed
# EMB = ROOT / "embeddings"

# pubchem_meta = pd.read_parquet(EMB / "pubchem_metadata.parquet")
# smiles_list = pubchem_meta["SMILES"].dropna().astype(str).tolist()

# print("Total SMILES:", len(smiles_list))
# # smiles_list = smiles_list[:20000]   # fits in 8 GB RAM


Total SMILES: 196185


In [ ]:
# def smiles_to_selfies(sm):
#     try:
#         return sf.encoder(sm)
#     except:
#         return None

# selfies_list = []
# for sm in smiles_list:
#     sf_sm = smiles_to_selfies(sm)
#     if sf_sm:
#         selfies_list.append(sf_sm)

# print("SELFIES count:", len(selfies_list))


SELFIES count: 193060


In [ ]:
# tokens = set()
# for s in selfies_list:
#     tokens.update(sf.split_selfies(s))

# tokens = sorted(list(tokens))
# tokens.append("[EOS]")

# token2idx = {t:i for i,t in enumerate(tokens)}
# idx2token = {i:t for t,i in token2idx.items()}

# vocab_size = len(token2idx)
# print("Vocab size:", vocab_size)


Vocab size: 481


In [ ]:
# import torch

# def tensorize(sf_str):
#     toks = list(sf.split_selfies(sf_str)) + ["[EOS]"]
#     return torch.tensor([token2idx[t] for t in toks], dtype=torch.long)

# train_data = [tensorize(s) for s in selfies_list]


In [ ]:
# import torch.nn as nn

# class SelfiesLSTM(nn.Module):
#     def __init__(self, vocab, emb=128, hidden=256):
#         super().__init__()
#         self.emb = nn.Embedding(vocab, emb)
#         self.lstm = nn.LSTM(emb, hidden, batch_first=True)
#         self.fc = nn.Linear(hidden, vocab)

#     def forward(self, x, h=None):
#         x = self.emb(x)
#         out, h = self.lstm(x, h)
#         return self.fc(out), h


In [ ]:
# model = SelfiesLSTM(vocab_size)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
# criterion = nn.CrossEntropyLoss()

# def train_epoch():
#     model.train()
#     total = 0
#     for x in train_data:
#         inp = x[:-1].unsqueeze(0)
#         tgt = x[1:]

#         optimizer.zero_grad()
#         out, _ = model(inp)
#         loss = criterion(out.view(-1, vocab_size), tgt)
#         loss.backward()
#         optimizer.step()
#         total += loss.item()

#     return total / len(train_data)

# for epoch in range(5):
#     loss = train_epoch()
#     print(f"Epoch {epoch+1} | Loss = {loss:.4f}")


In [8]:
# Cell 1 — Imports
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import QED, Descriptors
import faiss
import random
import selfies as sf

ROOT = Path("..")
DATA = ROOT / "data"
EMB = ROOT / "embeddings"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)


Using: cpu


In [9]:
# Cell 2 — Load FAISS SMILES Index
pubchem_index = faiss.read_index(str(EMB / "pubchem_index.faiss"))
pubchem_meta = pd.read_parquet(EMB / "pubchem_metadata.parquet")

print("PubChem vectors:", pubchem_index.ntotal)


PubChem vectors: 192783


In [10]:
# Cell 3 — SMILES → SELFIES
smiles_list = pubchem_meta["SMILES"].dropna().tolist()

selfies_list = []
for sm in smiles_list:
    try:
        selfies_list.append(sf.encoder(sm))
    except:
        pass

print("Total SELFIES:", len(selfies_list))


Total SELFIES: 193060


In [11]:
# Cell 4 — Vocabulary
alphabet = set()
for s in selfies_list:
    alphabet.update(sf.split_selfies(s))

alphabet = sorted(list(alphabet))
alphabet.append("[EOS]")

token2idx = {t:i for i,t in enumerate(alphabet)}
idx2token = {i:t for t,i in token2idx.items()}
vocab_size = len(alphabet)

print("SELFIES vocab size:", vocab_size)


SELFIES vocab size: 481


In [12]:
# Cell 5 — SELFIES LSTM
class SelfiesLSTM(nn.Module):
    def __init__(self, vocab_size, hidden=256, layers=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, hidden, layers, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden


model = SelfiesLSTM(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()


In [13]:
# Cell 6
def tensorize_selfies(s):
    tokens = list(sf.split_selfies(s)) + ["[EOS]"]  # ✅ FIX
    return torch.tensor([token2idx[t] for t in tokens], dtype=torch.long)


train_data = selfies_list[:20000]   # 20k is ideal


In [14]:
# Cell 7
def train_epoch():
    model.train()
    total = 0

    for s in train_data:
        x = tensorize_selfies(s)

        inp = x[:-1].unsqueeze(0).to(device)
        tgt = x[1:].to(device)

        optimizer.zero_grad()
        out, _ = model(inp)
        loss = criterion(out.reshape(-1, vocab_size), tgt)
        loss.backward()
        optimizer.step()

        total += loss.item()

    return total / len(train_data)



for epoch in range(5):
    loss = train_epoch()
    print(f"Epoch {epoch+1}, Loss = {loss:.4f}")


Epoch 1, Loss = 1.3805
Epoch 2, Loss = 1.0100
Epoch 3, Loss = 0.9456
Epoch 4, Loss = 0.9060
Epoch 5, Loss = 0.8806


In [15]:
import torch
import random
import selfies as sf

def generate_selfies_lstm(
    model, token2idx, idx2token,
    seed="[C]", max_len=40, temperature=0.8
):
    model.eval()

    tokens = list(sf.split_selfies(seed))
    idxs = [token2idx[t] for t in tokens if t in token2idx]

    x = torch.tensor([idxs], dtype=torch.long)
    hidden = None

    generated = tokens.copy()

    for _ in range(max_len):
        out, hidden = model(x, hidden)
        logits = out[0, -1] / temperature
        probs = torch.softmax(logits, dim=0)

        idx = torch.multinomial(probs, 1).item()
        tok = idx2token[idx]

        if tok == "[EOS]":
            break

        generated.append(tok)
        x = torch.tensor([[idx]])

    return "".join(generated)


In [16]:
N = 200

generated_smiles = []
for _ in range(N):
    sf_str = generate_selfies_lstm(model, token2idx, idx2token)
    sm = sf.decoder(sf_str)
    if sm:
        generated_smiles.append(sm)

print("Generated:", len(generated_smiles))


Generated: 200


In [17]:
from rdkit import Chem
from rdkit.Chem import QED, Descriptors

def evaluate(smiles_list, train_smiles_set):
    rows = []
    for sm in smiles_list:
        mol = Chem.MolFromSmiles(sm)
        if mol:
            rows.append({
                "SMILES": sm,
                "valid": True,
                "QED": QED.qed(mol),
                "lipinski": (
                    Descriptors.MolWt(mol) <= 500 and
                    Descriptors.MolLogP(mol) <= 5 and
                    Descriptors.NumHDonors(mol) <= 5 and
                    Descriptors.NumHAcceptors(mol) <= 10
                ),
                "novel": sm not in train_smiles_set
            })
        else:
            rows.append({
                "SMILES": sm,
                "valid": False,
                "QED": None,
                "lipinski": False,
                "novel": False
            })

    df = pd.DataFrame(rows)

    return {
        "Total generated": len(df),
        "Validity (%)": df.valid.mean() * 100,
        "Uniqueness (%)": df.SMILES.nunique() / len(df) * 100,
        "Lipinski pass (%)": df[df.valid].lipinski.mean() * 100,
        "Novelty (%)": df[df.valid].novel.mean() * 100,
        "Avg QED": df[df.valid].QED.mean()
    }


In [18]:
train_smiles_set = set(smiles_list)

metrics = evaluate(generated_smiles, train_smiles_set)

for k, v in metrics.items():
    print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")


Total generated: 200
Validity (%): 100.00
Uniqueness (%): 94.00
Lipinski pass (%): 98.00
Novelty (%): 99.00
Avg QED: 0.51


In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

torch.save(
    {
        "model": model.state_dict(),
        "token2idx": token2idx,
        "idx2token": idx2token
    },
    MODEL_DIR / "selfies_lstm.pt"
)

print("Saved models/selfies_lstm.pt")


Saved models/selfies_lstm.pt
